<a href="https://colab.research.google.com/github/premasr351/prompt_quality_scoring_agent/blob/main/travel_planning_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
#!pip install langchain-google-genai
#STEP 1 - Import libraries
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from google.colab import userdata



In [27]:
#STEP2 - Load api key
os.environ["GOOGLE_API_KEY"] = userdata.get('GEMINI_API_KEY')

#STEP 3 - Initialize LLM
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [28]:
# Build Prompt

system_prompt = """
You are an expert travel agent.

1. You should only answer travel related questions (flights, hotels,destinations, trips, travel tips)
2. For any non travel related questions, respond with "I can't help with that"
"""


#travel agent prompt
travelagent_prompt_template = ChatPromptTemplate.from_messages(
    [
        SystemMessage(content=system_prompt),
        MessagesPlaceholder(variable_name="chat_history"),
    ]
)


#Summary Prompt
summary_prompt_template = ChatPromptTemplate.from_messages(
    [
        "system", "You are a helpful assistant that summarizes travel related questions.",
        "human", "Summarize the following travel related question: {history}",
    ]
)



In [32]:
chat_history = []

def get_llm_response(user_message, history):
    global chat_history # Use global to modify the main chat_history list

    # Add user message to history
    chat_history.append(HumanMessage(content=user_message))

    # Decide which prompt to use based on the number of messages for summarization
    if len(chat_history) % 5 == 0: # Summarize every 5 user messages (10 total messages in history: 5 human, 5 AI)
        summary_prompt_value = summary_prompt_template.format_prompt(history=chat_history)
        summary_response = llm.invoke(summary_prompt_value)
        print(f"\n--- SUMMARY (Chat history length: {len(chat_history)}, Modulo 5: {len(chat_history) % 5}) ---")
        print(summary_response.content)
        print("-----------------\n")

        # Clear chat history after summarization if desired, or keep it for continued context
        # For this example, let's keep it, but a real use case might clear it.

    # Always use travel agent prompt for the main conversation
    travelagent_prompt_value = travelagent_prompt_template.format_prompt(chat_history=chat_history)

    # Removed non-streaming approach, commenting it out as requested
    # response = llm.invoke(travelagent_prompt_value)

    full_ai_response_content = ""
    print("\nAI: ", end="") # Start AI response with a prefix
    for chunk in llm.stream(travelagent_prompt_value):
        print(chunk.content, end="", flush=True) # Print chunks as they arrive
        full_ai_response_content += chunk.content
    print() # Newline after the streamed response

    # Removed non-streaming approach, commenting it out as requested
    # chat_history.append(AIMessage(content=response.content))

    # Add the full AI response to chat_history from the streamed content
    chat_history.append(AIMessage(content=full_ai_response_content))

    return full_ai_response_content # Return the full accumulated content

Now you can use the `get_llm_response` function to interact with the LLM and manage the chat history. The chat history will be summarized every 5 user messages.

In [ ]:
# Example usage:
print(get_llm_response("Hello, I'd like to plan a trip.", chat_history))
print(get_llm_response("Where should I go for a relaxing beach vacation in Europe?", chat_history))
print(get_llm_response("What are some good hotels in Santorini?", chat_history))
print(get_llm_response("Tell me about the best time to visit Greece.", chat_history))
print(get_llm_response("How do I get from Athens to Santorini?", chat_history))
print(chat_history)
# The next message will trigger a summary
print(get_llm_response("Can you suggest some activities in Santorini?", chat_history))

In [18]:
print(chat_history)

[HumanMessage(content="Hello, I'd like to plan a trip.", additional_kwargs={}, response_metadata={}), AIMessage(content="Hello! I'd be happy to help you plan your trip. To get started, could you tell me a little more about what you're looking for? For example:\n\n*   **Where are you thinking of going?** (Or if you don't have a destination yet, what kind of place are you interested in – e.g., beach, city, mountains, adventure, relaxation?)\n*   **When are you planning to travel?** (Specific dates or a general time of year)\n*   **How long would you like your trip to be?**\n*   **Who are you traveling with?** (Solo, family, couple, friends)\n*   **What's your approximate budget?** (e.g., budget-friendly, mid-range, luxury)\n*   **What are your main interests for this trip?** (e.g., culture, food, nightlife, nature, history, shopping, adventure sports)\n\nThe more details you can provide, the better I can tailor a plan for you!", additional_kwargs={}, response_metadata={}, tool_calls=[], 